In [52]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import os

In [53]:
df = pd.read_csv("synthetic_fraud_dataset.csv")

print("Dataset shape:", df.shape)

Dataset shape: (50000, 14)


In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Transaction_ID                50000 non-null  object 
 1   User_ID                       50000 non-null  object 
 2   Transaction_Amount            50000 non-null  float64
 3   Transaction_Type              50000 non-null  object 
 4   Date                          50000 non-null  object 
 5   Account_Balance               50000 non-null  float64
 6   Device_Type                   50000 non-null  object 
 7   Location                      50000 non-null  object 
 8   Merchant_Category             50000 non-null  object 
 9   Previous_Fraudulent_Activity  50000 non-null  int64  
 10  Daily_Transaction_Count       50000 non-null  int64  
 11  Card_Type                     50000 non-null  object 
 12  Card_Age                      50000 non-null  int64  
 13  F

# Remove Whitespace from Text Columns

In [55]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
)

print(df.columns.tolist())

['Transaction_ID', 'User_ID', 'Transaction_Amount', 'Transaction_Type', 'Date', 'Account_Balance', 'Device_Type', 'Location', 'Merchant_Category', 'Previous_Fraudulent_Activity', 'Daily_Transaction_Count', 'Card_Type', 'Card_Age', 'Fraud_Label']


In [56]:
df.dtypes

Transaction_ID                   object
User_ID                          object
Transaction_Amount              float64
Transaction_Type                 object
Date                             object
Account_Balance                 float64
Device_Type                      object
Location                         object
Merchant_Category                object
Previous_Fraudulent_Activity      int64
Daily_Transaction_Count           int64
Card_Type                        object
Card_Age                          int64
Fraud_Label                       int64
dtype: object

In [57]:
df["Date"] = pd.to_datetime(
    df["Date"],
    format="%d %B %Y",
    errors="coerce"
)

In [58]:
print("Total Users:", df["User_ID"].count())
print("Unique Users:", df["User_ID"].nunique())

Total Users: 50000
Unique Users: 8963


# Validate Categorical Values

In [59]:
categorical_columns = [
    "Transaction_Type",
    "Device_Type",
    "Location",
    "Merchant_Category",
    "Card_Type"
]

for column in categorical_columns:
    print("\n" + "=" * 60)
    print(column)
    print("=" * 60)
    print(df[column].value_counts())


Transaction_Type
Transaction_Type
POS               12549
Online            12546
ATM Withdrawal    12453
Bank Transfer     12452
Name: count, dtype: int64

Device_Type
Device_Type
Tablet    16779
Mobile    16640
Laptop    16581
Name: count, dtype: int64

Location
Location
Tokyo       10208
Mumbai       9994
London       9945
Sydney       9938
New York     9915
Name: count, dtype: int64

Merchant_Category
Merchant_Category
Clothing       10033
Groceries      10019
Travel         10015
Restaurants     9976
Electronics     9957
Name: count, dtype: int64

Card_Type
Card_Type
Mastercard    12693
Visa          12560
Amex          12419
Discover      12328
Name: count, dtype: int64


# Validate Fraud Label

In [60]:
print(df["Fraud_Label"].value_counts())
print(
    "Invalid labels:",
    (~df["Fraud_Label"].isin([0, 1])).sum()
)

Fraud_Label
0    33933
1    16067
Name: count, dtype: int64
Invalid labels: 0


# Numerical Validation

In [61]:
print(
    "Negative Transaction Amounts:",
    (df["Transaction_Amount"] < 0).sum()
)

print(
    "Zero Transaction Amounts:",
    (df["Transaction_Amount"] == 0).sum()
)

print(
    "Negative Account Balances:",
    (df["Account_Balance"] < 0).sum()
)

print(
    "Invalid Daily Transaction Counts:",
    (df["Daily_Transaction_Count"] <= 0).sum()
)

print(
    "Invalid Card Ages:",
    (df["Card_Age"] <= 0).sum()
)

Negative Transaction Amounts: 0
Zero Transaction Amounts: 2
Negative Account Balances: 0
Invalid Daily Transaction Counts: 0
Invalid Card Ages: 0


In [62]:
zero_amount_transactions = df[
    df["Transaction_Amount"] == 0
]

zero_amount_transactions

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Date,Account_Balance,Device_Type,Location,Merchant_Category,Previous_Fraudulent_Activity,Daily_Transaction_Count,Card_Type,Card_Age,Fraud_Label
9775,TXN_12945,USER_8426,0.0,POS,2023-08-01,92945.09,Tablet,London,Electronics,0,13,Discover,209,1
17233,TXN_10863,USER_8798,0.0,POS,2023-02-16,79882.97,Mobile,Tokyo,Restaurants,0,14,Discover,230,1


# Outlier Detection

In [63]:
def detect_outliers_iqr(data, column):
    
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[
        (data[column] < lower_bound) |
        (data[column] > upper_bound)
    ]
    
    return {
        "Column": column,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower_Bound": lower_bound,
        "Upper_Bound": upper_bound,
        "Outlier_Count": len(outliers),
        "Outlier_Percentage": len(outliers) / len(data) * 100
    }

In [64]:
numeric_columns = [
    "Transaction_Amount",
    "Account_Balance",
    "Daily_Transaction_Count",
    "Card_Age"
]

outlier_results = []

for column in numeric_columns:
    outlier_results.append(
        detect_outliers_iqr(df, column)
    )

outlier_report = pd.DataFrame(outlier_results)

outlier_report.round(2)

,Column,Q1,Q3,IQR,Lower_Bound,Upper_Bound,Outlier_Count,Outlier_Percentage
0,Transaction_Amount,28.68,138.85,110.17,-136.58,304.12,2260,4.52
1,Account_Balance,25356.00,75115.14,49759.14,-49282.72,149753.85,0,0.00
2,Daily_Transaction_Count,4.00,11.00,7.00,-6.50,21.50,0,0.00
3,Card_Age,60.00,180.00,120.00,-120.00,360.00,0,0.00


In [65]:
print(
    "Users with more than one transaction:",
    (user_transaction_counts > 1).sum()
)

print(
    "Maximum transactions by one user:",
    user_transaction_counts.max()
)

Users with more than one transaction: 8768
Maximum transactions by one user: 16


# Create Date Features

In [66]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Month_Name"] = df["Date"].dt.month_name()
df["Day"] = df["Date"].dt.day
df["Day_of_Week"] = df["Date"].dt.dayofweek
df["Day_Name"] = df["Date"].dt.day_name()

In [67]:
df[
    [
        "Date",
        "Year",
        "Month",
        "Month_Name",
        "Day",
        "Day_of_Week",
        "Day_Name"
    ]
].head()

,Date,Year,Month,Month_Name,Day,Day_of_Week,Day_Name
0,2023-08-14,2023,8,August,14,0,Monday
1,2023-06-07,2023,6,June,7,2,Wednesday
2,2023-06-20,2023,6,June,20,1,Tuesday
3,2023-12-07,2023,12,December,7,3,Thursday
4,2023-11-11,2023,11,November,11,5,Saturday


In [68]:
print("Years:", df["Year"].unique())
print("Months:", sorted(df["Month"].unique()))
print("Days of week:", sorted(df["Day_of_Week"].unique()))

Years: [2023]
Months: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Days of week: [0, 1, 2, 3, 4, 5, 6]


In [69]:
df.dtypes

Transaction_ID                          object
User_ID                                 object
Transaction_Amount                     float64
Transaction_Type                        object
Date                            datetime64[ns]
Account_Balance                        float64
Device_Type                             object
Location                                object
Merchant_Category                       object
Previous_Fraudulent_Activity             int64
Daily_Transaction_Count                  int64
Card_Type                               object
Card_Age                                 int64
Fraud_Label                              int64
Year                                     int32
Month                                    int32
Month_Name                              object
Day                                      int32
Day_of_Week                              int32
Day_Name                                object
dtype: object

# Final Data Quality Report

In [70]:
df.head()

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Date,Account_Balance,Device_Type,Location,Merchant_Category,Previous_Fraudulent_Activity,Daily_Transaction_Count,Card_Type,Card_Age,Fraud_Label,Year,Month,Month_Name,Day,Day_of_Week,Day_Name
0,TXN_33553,USER_1834,39.79,POS,2023-08-14,93213.17,Laptop,Sydney,Travel,0,7,Amex,65,0,2023,8,August,14,0,Monday
1,TXN_9427,USER_7875,1.19,Bank Transfer,2023-06-07,75725.25,Mobile,New York,Clothing,0,13,Mastercard,186,1,2023,6,June,7,2,Wednesday
2,TXN_199,USER_2734,28.96,Online,2023-06-20,1588.96,Tablet,Mumbai,Restaurants,0,14,Visa,226,1,2023,6,June,20,1,Tuesday
3,TXN_12447,USER_2617,254.32,ATM Withdrawal,2023-12-07,76807.20,Tablet,New York,Clothing,0,8,Visa,76,1,2023,12,December,7,3,Thursday
4,TXN_39489,USER_2014,31.28,POS,2023-11-11,92354.66,Mobile,Mumbai,Electronics,1,14,Mastercard,140,1,2023,11,November,11,5,Saturday


In [71]:
df.tail()

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Date,Account_Balance,Device_Type,Location,Merchant_Category,Previous_Fraudulent_Activity,Daily_Transaction_Count,Card_Type,Card_Age,Fraud_Label,Year,Month,Month_Name,Day,Day_of_Week,Day_Name
49995,TXN_11284,USER_4796,45.05,Online,2023-01-29,76960.11,Mobile,Tokyo,Clothing,0,2,Amex,98,0,2023,1,January,29,6,Sunday
49996,TXN_44732,USER_1171,126.15,POS,2023-05-09,28791.75,Mobile,Tokyo,Clothing,0,13,Visa,93,1,2023,5,May,9,1,Tuesday
49997,TXN_38158,USER_2510,72.02,Online,2023-01-30,29916.41,Laptop,Mumbai,Clothing,1,1,Visa,114,0,2023,1,January,30,0,Monday
49998,TXN_860,USER_2248,64.89,Bank Transfer,2023-03-09,67895.67,Mobile,Tokyo,Electronics,0,13,Discover,72,1,2023,3,March,9,3,Thursday
49999,TXN_15795,USER_6529,13.00,Bank Transfer,2023-08-19,7668.82,Tablet,London,Restaurants,0,5,Mastercard,154,1,2023,8,August,19,5,Saturday


In [72]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 20 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Transaction_ID                50000 non-null  object        
 1   User_ID                       50000 non-null  object        
 2   Transaction_Amount            50000 non-null  float64       
 3   Transaction_Type              50000 non-null  object        
 4   Date                          50000 non-null  datetime64[ns]
 5   Account_Balance               50000 non-null  float64       
 6   Device_Type                   50000 non-null  object        
 7   Location                      50000 non-null  object        
 8   Merchant_Category             50000 non-null  object        
 9   Previous_Fraudulent_Activity  50000 non-null  int64         
 10  Daily_Transaction_Count       50000 non-null  int64         
 11  Card_Type                   

In [73]:
print("=" * 70)
print("FINAL DATA QUALITY REPORT")
print("=" * 70)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print(
    "Missing values:",
    df.isnull().sum().sum()
)

print(
    "Duplicate rows:",
    df.duplicated().sum()
)

print(
    "Duplicate Transaction IDs:",
    df["Transaction_ID"].duplicated().sum()
)

print(
    "Invalid Fraud Labels:",
    (~df["Fraud_Label"].isin([0, 1])).sum()
)

print(
    "Invalid Dates:",
    df["Date"].isna().sum()
)

print(
    "Negative Transaction Amounts:",
    (df["Transaction_Amount"] < 0).sum()
)

print(
    "Negative Account Balances:",
    (df["Account_Balance"] < 0).sum()
)

print(
    "Invalid Daily Transaction Counts:",
    (df["Daily_Transaction_Count"] <= 0).sum()
)

print(
    "Invalid Card Ages:",
    (df["Card_Age"] <= 0).sum()
)

print("=" * 70)

FINAL DATA QUALITY REPORT
Rows: 50000
Columns: 20
Missing values: 0
Duplicate rows: 0
Duplicate Transaction IDs: 0
Invalid Fraud Labels: 0
Invalid Dates: 0
Negative Transaction Amounts: 0
Negative Account Balances: 0
Invalid Daily Transaction Counts: 0
Invalid Card Ages: 0


In [74]:
df.to_csv("cleaned_fraud_data.csv", index=False)